# Teradata SQL Templates Generator - Examples

This notebook demonstrates how to use the `teradata_sql_utils` module to generate Teradata SQL templates for:
- Adding date filters to queries
- Creating volatile tables
- Working with QueryGrid
- Date partitioning queries

Let's first import the module. (Make sure `teradata_sql_utils.py` is in the same directory as this notebook.)

In [1]:
import teradata_sql_utils as tdu

# Helper function to print SQL more nicely
def print_sql(sql_str, title=None):
    if title:
        print(f"=== {title} ===")
    print(sql_str)
    print("\n" + "-"*80 + "\n")

## 1. Adding Date Filters to Queries

Let's start with a simple example: adding date filters to an existing SQL query. This is useful when you need to dynamically limit query results to specific date ranges.

In [ ]:
# Base query with no WHERE clause
base_query_1 = """
SELECT
    customer_id,
    order_id,
    order_amount
FROM sales.customer_orders
"""

# Add date filters
filtered_query = tdu.create_date_filtered_query(
    base_query=base_query_1,
    date_column="order_date",
    start_date="2023-01-01",
    end_date="2023-03-31"
)

print_sql(filtered_query, "Query with Date Filters Added")

=== Query with Date Filters Added ===
SELECT 
    customer_id,
    order_id,
    order_amount
FROM sales.customer_orders WHERE order_date >= '2023-01-01' AND order_date <= '2023-03-31'

--------------------------------------------------------------------------------



### Adding Date Filters to a Query with Existing WHERE Clause

Let's see how it handles a query that already has a WHERE clause:

In [ ]:
# Base query with existing WHERE clause
base_query_2 = """
SELECT
    customer_id,
    order_id,
    order_amount
FROM sales.customer_orders
WHERE order_status = 'Completed'
"""

# Add date filters
filtered_query_2 = tdu.create_date_filtered_query(
    base_query=base_query_2,
    date_column="order_date",
    start_date="2023-01-01",
    end_date="2023-03-31"
)

print_sql(filtered_query_2, "Query with Date Filters Added to Existing WHERE")

=== Query with Date Filters Added to Existing WHERE ===
SELECT 
    customer_id,
    order_id,
    order_amount
FROM sales.customer_orders
WHERE order_date >= '2023-01-01' AND order_date <= '2023-03-31' AND  order_status = 'Completed'

--------------------------------------------------------------------------------



### Adding Date Filters to a Query with GROUP BY

Let's see how it handles a query with a GROUP BY clause:

In [ ]:
# Base query with GROUP BY
base_query_3 = """
SELECT
    customer_id,
    COUNT(order_id) AS total_orders,
    SUM(order_amount) AS total_spent
FROM sales.customer_orders
GROUP BY customer_id
"""

# Add date filters
filtered_query_3 = tdu.create_date_filtered_query(
    base_query=base_query_3,
    date_column="order_date",
    start_date="2023-01-01",
    end_date="2023-03-31"
)

print_sql(filtered_query_3, "Query with Date Filters Added Before GROUP BY")

=== Query with Date Filters Added Before GROUP BY ===
SELECT 
    customer_id,
    COUNT(order_id) AS total_orders,
    SUM(order_amount) AS total_spent
FROM sales.customer_orders
WHERE order_date >= '2023-01-01' AND order_date <= '2023-03-31' GROUP BY customer_id

--------------------------------------------------------------------------------



## 2. Creating Volatile Tables

Now let's see how to create volatile tables from a query. The `create_volatile_table_sql()` function handles:
- Checking if the table exists and dropping it
- Creating the table with proper indexing
- Collecting statistics for query optimization

In [ ]:
# Sample query to populate the volatile table
query = """
SELECT
    c.customer_id,
    c.customer_name,
    c.customer_email,
    c.signup_date,
    COUNT(o.order_id) AS order_count,
    SUM(o.order_amount) AS total_spent
FROM customer_database.customers c
LEFT JOIN sales.orders o ON c.customer_id = o.customer_id
WHERE o.order_date >= '2023-01-01'
GROUP BY c.customer_id, c.customer_name, c.customer_email, c.signup_date
"""

# Generate SQL to create volatile table
volatile_sql = tdu.create_volatile_table_sql(
    table_name="vol_customer_spending",
    query=query,
    primary_index_cols=["customer_id"],
    check_exists=True,
    collect_stats=True
)

print_sql(volatile_sql, "Creating a Volatile Table")

=== Creating a Volatile Table ===

-- Check if table exists and drop it
BEGIN
    DECLARE CONTINUE HANDLER FOR SQLSTATE '42000' BEGIN END;
    DROP TABLE vol_customer_spending;
END;


-- Create volatile table
CREATE VOLATILE TABLE vol_customer_spending AS (
    
SELECT 
    c.customer_id,
    c.customer_name,
    c.customer_email,
    c.signup_date,
    COUNT(o.order_id) AS order_count,
    SUM(o.order_amount) AS total_spent
FROM customer_database.customers c
LEFT JOIN sales.orders o ON c.customer_id = o.customer_id
WHERE o.order_date >= '2023-01-01'
GROUP BY c.customer_id, c.customer_name, c.customer_email, c.signup_date

) WITH DATA PRIMARY INDEX (customer_id)
ON COMMIT PRESERVE ROWS;


-- Collect statistics for query optimization
COLLECT STATISTICS ON vol_customer_spending;


--------------------------------------------------------------------------------



### Using a String for Primary Index

You can also pass a string for the primary index if you have a specific expression:

In [6]:
# Generate SQL with string-defined primary index
volatile_sql_2 = tdu.create_volatile_table_sql(
    table_name="vol_daily_sales",
    query="SELECT store_id, sale_date, SUM(amount) AS daily_total FROM sales.transactions GROUP BY store_id, sale_date",
    primary_index_cols="store_id, CAST(sale_date AS DATE)",  # Using string format for complex index
    check_exists=True,
    collect_stats=True
)

print_sql(volatile_sql_2, "Volatile Table with String-Defined Primary Index")

=== Volatile Table with String-Defined Primary Index ===

-- Check if table exists and drop it
BEGIN
    DECLARE CONTINUE HANDLER FOR SQLSTATE '42000' BEGIN END;
    DROP TABLE vol_daily_sales;
END;


-- Create volatile table
CREATE VOLATILE TABLE vol_daily_sales AS (
    SELECT store_id, sale_date, SUM(amount) AS daily_total FROM sales.transactions GROUP BY store_id, sale_date
) WITH DATA PRIMARY INDEX (store_id, CAST(sale_date AS DATE))
ON COMMIT PRESERVE ROWS;


-- Collect statistics for query optimization
COLLECT STATISTICS ON vol_daily_sales;


--------------------------------------------------------------------------------



### Without Checking Existence and Stats Collection

You can also skip the existence check and stats collection if needed:

In [7]:
# Generate SQL without checks or stats
volatile_sql_3 = tdu.create_volatile_table_sql(
    table_name="vol_simple_lookup",
    query="SELECT product_id, product_name, price FROM products WHERE active = 'Y'",
    primary_index_cols=["product_id"],
    check_exists=False,  # Skip existence check
    collect_stats=False  # Skip stats collection
)

print_sql(volatile_sql_3, "Simple Volatile Table Creation")

=== Simple Volatile Table Creation ===

-- Create volatile table
CREATE VOLATILE TABLE vol_simple_lookup AS (
    SELECT product_id, product_name, price FROM products WHERE active = 'Y'
) WITH DATA PRIMARY INDEX (product_id)
ON COMMIT PRESERVE ROWS;


--------------------------------------------------------------------------------



## 3. QueryGrid Volatile Tables

QueryGrid allows executing a query on a remote Teradata database and bringing the results to the local system. The function wraps this process in the correct syntax.

In [ ]:
# Sample query to execute on remote system
remote_query = """
SELECT
    product_id,
    product_name,
    category,
    department,
    price,
    cost,
    inventory_level
FROM inventory.product_catalog
WHERE active_flag = 'Y'
  AND inventory_level > 0
"""

# Generate SQL to create volatile table from QueryGrid
querygrid_sql = tdu.create_volatile_table_sql(
    table_name="vol_available_products",
    query=remote_query,
    primary_index_cols=["product_id"],
    is_querygrid=True,
    target_database="CENTRAL_EDW"
)

print_sql(querygrid_sql, "QueryGrid Volatile Table Creation")

=== QueryGrid Volatile Table Creation ===

-- Check if table exists and drop it
BEGIN
    DECLARE CONTINUE HANDLER FOR SQLSTATE '42000' BEGIN END;
    DROP TABLE vol_available_products;
END;


-- Create volatile table using QueryGrid
CREATE VOLATILE TABLE vol_available_products AS (
    SELECT * FROM (
        EXECUTE IMMEDIATE
        $$
            
SELECT 
    product_id,
    product_name,
    category,
    department,
    price,
    cost,
    inventory_level
FROM inventory.product_catalog
WHERE active_flag = 'Y'
  AND inventory_level > 0

        $$
        ON CENTRAL_EDW
    ) AS QueryGridResult
) WITH DATA PRIMARY INDEX (product_id)
ON COMMIT PRESERVE ROWS;


-- Collect statistics for query optimization
COLLECT STATISTICS ON vol_available_products;


--------------------------------------------------------------------------------



## 4. Date Partitioned Queries

Date partitioning can improve performance for large datasets by grouping data by time periods. The `create_date_partitioned_query()` function lets you partition by day, month, or year.

In [ ]:
# Sample base query
base_query = """
SELECT
    store_id,
    product_id,
    SUM(quantity) AS units_sold,
    SUM(sales_amount) AS total_sales
FROM sales.daily_transactions
"""

# Create a month-partitioned query
month_partitioned = tdu.create_date_partitioned_query(
    base_query=base_query,
    date_column="transaction_date",
    partition_by="month",
    start_date="2023-01-01",
    end_date="2023-12-31"
)

print_sql(month_partitioned, "Month-Partitioned Query")

=== Month-Partitioned Query ===
SELECT 
    store_id,
    product_id,
    SUM(quantity) AS units_sold,
    SUM(sales_amount) AS total_sales
FROM sales.daily_transactions WHERE transaction_date >= '2023-01-01' AND transaction_date <= '2023-12-31' GROUP BY EXTRACT(YEAR FROM transaction_date) * 100 + EXTRACT(MONTH FROM transaction_date)

--------------------------------------------------------------------------------



In [10]:
# Create a day-partitioned query
day_partitioned = tdu.create_date_partitioned_query(
    base_query=base_query,
    date_column="transaction_date",
    partition_by="day",
    start_date="2023-03-01",
    end_date="2023-03-31"
)

print_sql(day_partitioned, "Day-Partitioned Query")

=== Day-Partitioned Query ===
SELECT 
    store_id,
    product_id,
    SUM(quantity) AS units_sold,
    SUM(sales_amount) AS total_sales
FROM sales.daily_transactions WHERE transaction_date >= '2023-03-01' AND transaction_date <= '2023-03-31' GROUP BY CAST(transaction_date AS DATE)

--------------------------------------------------------------------------------



In [11]:
# Create a year-partitioned query
year_partitioned = tdu.create_date_partitioned_query(
    base_query=base_query,
    date_column="transaction_date",
    partition_by="year",
    start_date="2020-01-01",
    end_date="2023-12-31"
)

print_sql(year_partitioned, "Year-Partitioned Query")

=== Year-Partitioned Query ===
SELECT 
    store_id,
    product_id,
    SUM(quantity) AS units_sold,
    SUM(sales_amount) AS total_sales
FROM sales.daily_transactions WHERE transaction_date >= '2020-01-01' AND transaction_date <= '2023-12-31' GROUP BY EXTRACT(YEAR FROM transaction_date)

--------------------------------------------------------------------------------



## 5. Putting It All Together: Complete Workflow

Let's create a complete workflow that:
1. Filters a base query by date
2. Creates a volatile table from the filtered query
3. Saves the SQL to a file for later execution

In [ ]:
# Step 1: Start with a base query
base_query = """
SELECT
    c.customer_id,
    c.customer_name,
    c.segment,
    COUNT(DISTINCT o.order_id) AS order_count,
    SUM(o.order_amount) AS total_spent,
    AVG(o.order_amount) AS avg_order_value
FROM customers c
JOIN orders o ON c.customer_id = o.customer_id
WHERE c.status = 'Active'
GROUP BY c.customer_id, c.customer_name, c.segment
"""

# Step 2: Add date filters to the query
filtered_query = tdu.create_date_filtered_query(
    base_query=base_query,
    date_column="o.order_date",
    start_date="2023-01-01",
    end_date="2023-12-31"
)

# Step 3: Generate SQL to create a volatile table
volatile_table_sql = tdu.create_volatile_table_sql(
    table_name="vol_customer_kpis_2023",
    query=filtered_query,
    primary_index_cols=["customer_id"],
    check_exists=True,
    collect_stats=True
)

print_sql(volatile_table_sql, "Complete Workflow SQL")

# Step 4: Save the SQL to a file (uncomment to save)
# with open("create_customer_kpis_volatile.sql", "w") as f:
#     f.write(volatile_table_sql)

=== Complete Workflow SQL ===

-- Check if table exists and drop it
BEGIN
    DECLARE CONTINUE HANDLER FOR SQLSTATE '42000' BEGIN END;
    DROP TABLE vol_customer_kpis_2023;
END;


-- Create volatile table
CREATE VOLATILE TABLE vol_customer_kpis_2023 AS (
    SELECT 
    c.customer_id,
    c.customer_name,
    c.segment,
    COUNT(DISTINCT o.order_id) AS order_count,
    SUM(o.order_amount) AS total_spent,
    AVG(o.order_amount) AS avg_order_value
FROM customers c
JOIN orders o ON c.customer_id = o.customer_id
WHERE o.order_date >= '2023-01-01' AND o.order_date <= '2023-12-31' AND  c.status = 'Active'
GROUP BY c.customer_id, c.customer_name, c.segment
) WITH DATA PRIMARY INDEX (customer_id)
ON COMMIT PRESERVE ROWS;


-- Collect statistics for query optimization
COLLECT STATISTICS ON vol_customer_kpis_2023;


--------------------------------------------------------------------------------



## 6. Working with Real Data

If you have a Teradata connection set up (e.g., using `teradatasql` or another driver), you can execute the generated SQL directly from Python:

In [13]:
# Example of how you might execute the SQL (pseudocode, requires actual connection)
"""
import teradatasql

# Create connection
conn = teradatasql.connect(host='your_teradata_server', user='your_username', password='your_password')
cursor = conn.cursor()

# Execute our generated SQL
cursor.execute(volatile_table_sql)

# Now we can query the volatile table
cursor.execute("SELECT * FROM vol_customer_kpis_2023 WHERE total_spent > 1000 ORDER BY total_spent DESC")
results = cursor.fetchall()

# Process results
for row in results:
    print(row)

# Close connection
cursor.close()
conn.close()
"""

'\nimport teradatasql\n\n# Create connection\nconn = teradatasql.connect(host=\'your_teradata_server\', user=\'your_username\', password=\'your_password\')\ncursor = conn.cursor()\n\n# Execute our generated SQL\ncursor.execute(volatile_table_sql)\n\n# Now we can query the volatile table\ncursor.execute("SELECT * FROM vol_customer_kpis_2023 WHERE total_spent > 1000 ORDER BY total_spent DESC")\nresults = cursor.fetchall()\n\n# Process results\nfor row in results:\n    print(row)\n\n# Close connection\ncursor.close()\nconn.close()\n'

## 7. Conclusion

In this notebook, we've demonstrated how to use the `teradata_sql_utils` module to:

1. Add date filters to existing SQL queries
2. Create volatile tables with proper indexing and statistics collection
3. Work with QueryGrid for cross-database queries
4. Partition queries by date periods (day, month, year)
5. Combine these functions in a complete workflow

These utilities can help streamline your Teradata SQL generation process, making it more efficient and consistent. The module is designed to be lightweight and easy to integrate into your existing Python workflows.